In [99]:
'''
1. parallel chain -> summarizing, sentiment analysis, key points extraction
2. combine into one and find overall analysis

'''

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from operator import itemgetter
from typing import Literal
from langchain_mistralai import ChatMistralAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableBranch, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser
from rich.console import Console
from rich.markdown import Markdown

In [100]:
load_dotenv()

llm = ChatMistralAI(
    model_name="mistral-small-latest",
    max_retries=2,
    max_tokens=1000
)

class Review(BaseModel):
    sentiment: Literal['positive', 'negative'] = Field(description="sentiment of an review [positive or negetive]")

str_parser = StrOutputParser()
pydantic_parser = PydanticOutputParser(pydantic_object=Review)
console = Console()

In [101]:


sentiment = PromptTemplate(
    template="Analyse sentiment of feedback: {feedback} \n {format_instruction}",
    input_variables=['feedback'],
    partial_variables={'format_instruction': pydantic_parser.get_format_instructions() }
)

positive_prompt = PromptTemplate(
    template=(
        "Write an appropiate response on this positive Feedback: {feedback}"
    ),
    input_variables=['feedback']
)

negative_prompt = PromptTemplate(
    template=(
        "Write an appropiate response on this negative Feedback: {feedback}:"
    ),
    input_variables=['feedback']
)

classifier_chain = RunnableParallel({
    "feedback": itemgetter("feedback"),
    "sentiment": sentiment | llm | pydantic_parser
})

# print(classifier_chain.invoke({"feedback": "given phone is not that good as expected"}))

branch_chain = RunnableBranch(
    (lambda x: x["sentiment"].sentiment == 'positive', positive_prompt | llm | str_parser),
    (lambda x: x["sentiment"].sentiment == 'negative', negetive_prompt | llm | str_parser),
    RunnableLambda(lambda x: "could not found sentiment")
)

chain = classifier_chain | branch_chain


print(chain.invoke({"feedback": "given phone is not that good as expected"}))
chain.get_graph().print_ascii()

Here are some professional and constructive ways to respond to this negative feedback, depending on the context (e.g., customer service, product review, etc.):

### **1. For Customer Service (Apologetic & Solution-Oriented)**
*"We’re really sorry to hear that your experience with our phone didn’t meet your expectations. Could you share more details about what didn’t meet your expectations? We’d love to make it right for you."*

### **2. For a Product Review (Apologetic & Responsive)**
*"We appreciate your feedback and are sorry to hear that the phone didn’t meet your expectations. Could you let us know what specific features or aspects didn’t meet your satisfaction? Your insights help us improve."*

### **3. For Social Media (Polite & Professional)**
*"Thanks for sharing your feedback. We’re always looking to improve, so we’d appreciate more details on what didn’t meet your expectations. Feel free to DM us for further assistance."*

### **4. For Internal Team Discussion (Analytical & A

In [102]:
summarize_prompt = PromptTemplate(
    template="Summarize the given news \n news: {news}",
    input_variables=['news']
)
sentiment_prompt = PromptTemplate(
    template="Summarize the given news \n news: {news}",
    input_variables=['news']
)
key_points_prompt = PromptTemplate(
    template="Summarize the given news \n news: {news}",
    input_variables=['news']
)
combination_prompt = PromptTemplate(
    template="Based on the text, its sentiment and key points give brief overview on it, \n text: {summarize} \n sentiment: {sentiment} \n key points: {key_points}"
)

parallel_chain = RunnableParallel({
    "summarize": summarize_prompt | llm | str_parser,
    "sentiment": sentiment_prompt | llm | str_parser,
    "key_points": key_points_prompt | llm |str_parser
})

chain = parallel_chain | combination_prompt | llm | str_parser

chain.get_graph().print_ascii()

                     +-----------------------------------------------+                       
                     | Parallel<summarize,sentiment,key_points>Input |                       
                     +-----------------------------------------------+                       
                         *******              *              *******                         
                    *****                     *                     *****                    
                ****                          *                          ****                
+----------------+                  +----------------+                  +----------------+   
| PromptTemplate |                  | PromptTemplate |                  | PromptTemplate |   
+----------------+                  +----------------+                  +----------------+   
          *                                   *                                   *          
          *                                   *             